# LST Regression Model + Scenario Simulation

Trains a model to **predict Land Surface Temperature (LST)** from urban morphology features — no satellite thermal input. The trained model is then used to simulate the LST reduction that green walls would cause by increasing local NDVI.

| | |
|---|---|
| **Target (y)** | `mean_LST` per 100×100m cell |
| **Features (X)** | NDVI, pct_green, pct_built, building height, building density, dist to road, dist to water |
| **Models** | Linear Regression → Random Forest → XGBoost |
| **Scenario** | Increase NDVI by green-wall delta → re-predict LST → compute cooling impact |

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import joblib
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb

OUT = Path('../outputs')
OUT.mkdir(exist_ok=True)

FEATURES = [
    'mean_NDVI', 'pct_green', 'pct_built',
    'mean_building_height', 'building_density',
    'dist_major_road', 'dist_to_water'
]
TARGET = 'mean_LST'
print('Setup OK')

## 1 — Load Data

In [ ]:
grid = gpd.read_file('../outputs/tel_aviv_grid_features.gpkg')
grid = grid.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)

print(f'Cells: {len(grid):,}')
print(f'LST range: {grid[TARGET].min():.1f}°C – {grid[TARGET].max():.1f}°C')
grid[FEATURES + [TARGET]].describe().round(2)

## 2 — Train / Test Split

In [ ]:
X = grid[FEATURES].values
y = grid[TARGET].values

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, grid.index, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train):,} cells  |  Test: {len(X_test):,} cells')

## 3 — Train Models

In [ ]:
models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=12, min_samples_leaf=5,
        n_jobs=-1, random_state=42
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'MAE' : mean_absolute_error(y_test, preds),
        'RMSE': root_mean_squared_error(y_test, preds),
        'R²'  : r2_score(y_test, preds),
        'preds': preds
    }
    print(f"{name:20s}  MAE={results[name]['MAE']:.2f}°C  RMSE={results[name]['RMSE']:.2f}°C  R²={results[name]['R²']:.3f}")

## 4 — Model Comparison

In [ ]:
metrics_df = pd.DataFrame({k: {m: v for m, v in v.items() if m != 'preds'}
                           for k, v in results.items()}).T.round(3)
display(metrics_df)

best_name = metrics_df['R²'].idxmax()
best_model = models[best_name]
print(f'\nBest model: {best_name}')

## 5 — Actual vs Predicted + Residuals Map

In [ ]:
best_preds_test = results[best_name]['preds']

# Predict on full grid for mapping
grid['predicted_LST'] = best_model.predict(X)
grid['residual']      = grid[TARGET] - grid['predicted_LST']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Actual vs predicted scatter
ax = axes[0]
ax.scatter(y_test, best_preds_test, alpha=0.4, s=10, color='steelblue')
lims = [min(y_test.min(), best_preds_test.min()), max(y_test.max(), best_preds_test.max())]
ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('Actual LST (°C)')
ax.set_ylabel('Predicted LST (°C)')
ax.set_title(f'{best_name}\nR²={results[best_name]["R²"]:.3f}  MAE={results[best_name]["MAE"]:.2f}°C')

# Predicted LST map
grid_wgs = grid.to_crs(4326)
grid_wgs.plot(column='predicted_LST', cmap='RdYlBu_r', legend=True,
              legend_kwds={'label':'Predicted LST (°C)', 'shrink':0.7}, ax=axes[1])
axes[1].set_title('Predicted LST')
axes[1].axis('off')

# Residuals map
grid_wgs.plot(column='residual', cmap='coolwarm', legend=True,
              legend_kwds={'label':'Residual (°C)', 'shrink':0.7}, ax=axes[2])
axes[2].set_title('Residuals (Actual − Predicted)')
axes[2].axis('off')

plt.tight_layout()
plt.savefig(OUT / 'model_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mname in zip(axes, ['Random Forest', 'XGBoost']):
    m = models[mname]
    imp = m.feature_importances_ if mname == 'Random Forest' else m.feature_importances_
    order = np.argsort(imp)
    ax.barh([FEATURES[i] for i in order], imp[order], color='steelblue')
    ax.set_title(f'{mname} — Feature Importance')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig(OUT / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 — Scenario Simulation: Green Wall Impact

For each grid cell, estimate the NDVI increase a green wall on surrounding buildings would cause,
then re-predict LST to get the predicted cooling impact.

In [ ]:
# Green wall NDVI delta:
# Assume green walls cover 40% of building perimeter × height in a cell.
# Additional green area / cell area → NDVI increase (capped at +0.25).
COVERAGE_RATIO = 0.4
CELL_AREA = 100 * 100   # m²

# Estimate perimeter from pct_built and building count
# Simple proxy: sqrt(footprint_area) × 4 per building
avg_footprint = (grid['pct_built'] * CELL_AREA / grid['building_density'].replace(0, np.nan)).fillna(0)
avg_perimeter = 4 * np.sqrt(avg_footprint.clip(1))

wall_area_per_cell = avg_perimeter * grid['mean_building_height'] * COVERAGE_RATIO * grid['building_density']
ndvi_delta = (wall_area_per_cell / CELL_AREA).clip(0, 0.25)

# Modify NDVI feature and re-predict
X_scenario = grid[FEATURES].copy()
X_scenario['mean_NDVI'] = (X_scenario['mean_NDVI'] + ndvi_delta).clip(upper=1.0)
X_scenario['pct_green'] = (X_scenario['pct_green'] + ndvi_delta * 0.5).clip(upper=1.0)

grid['predicted_LST_with_walls'] = best_model.predict(X_scenario.values)
grid['predicted_cooling']        = (grid['predicted_LST'] - grid['predicted_LST_with_walls']).clip(lower=0)

print(f"Mean predicted cooling : {grid['predicted_cooling'].mean():.2f}°C")
print(f"Max predicted cooling  : {grid['predicted_cooling'].max():.2f}°C")
print(f"Cells with >0.5°C gain : {(grid['predicted_cooling'] > 0.5).sum():,}")

In [ ]:
grid_wgs = grid.to_crs(4326)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

grid_wgs.plot(column='predicted_LST', cmap='RdYlBu_r', legend=True,
              legend_kwds={'label':'Predicted LST (°C)', 'shrink':0.7}, ax=axes[0])
axes[0].set_title('Predicted LST (baseline)', fontweight='bold')
axes[0].axis('off')

grid_wgs.plot(column='predicted_cooling', cmap='Blues', legend=True,
              legend_kwds={'label':'Predicted cooling (°C)', 'shrink':0.7}, ax=axes[1])
axes[1].set_title('Predicted LST Reduction from Green Walls', fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(OUT / 'scenario_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 — Save Model + Updated Grid

In [ ]:
joblib.dump(best_model, OUT / 'lst_model.pkl')
grid.to_file(OUT / 'tel_aviv_grid_features.gpkg', driver='GPKG')

print(f'Model saved : outputs/lst_model.pkl  ({best_name})')
print(f'Grid saved  : outputs/tel_aviv_grid_features.gpkg')
print(f'New columns : predicted_LST, residual, predicted_LST_with_walls, predicted_cooling')